# Stacking untuk Klasifikasi

Stacking menggabungkan beberapa model dasar, lalu memakai keluarannya sebagai masukan bagi model meta. Model meta mempelajari cara menggabungkan prediksi tersebut agar hasil akhirnya lebih baik daripada setiap model tunggal.

Agar model meta tidak belajar dari prediksi data yang sama, prediksi model dasar dibuat lewat validasi silang internal.

## Tujuan

- Membangun `StackingClassifier` dengan beberapa model dasar.
- Membandingkan akurasi latih dan uji.
- Menyetel hyperparameter model dasar lewat grid search.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import metrics, preprocessing
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


In [ ]:
def get_accuracy(X_train, X_test, y_train, y_test, model):
    return {
        "test Accuracy": metrics.accuracy_score(y_test, model.predict(X_test)),
        "train Accuracy": metrics.accuracy_score(y_train, model.predict(X_train)),
    }


## Data anggur

Data ini memuat hasil uji kimia anggur dari tiga kultivar. Kolom `Class` menjadi target klasifikasi.


In [ ]:
WINE_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/wine.data"
)
wine_data = pd.read_csv(
    WINE_URL,
    names=[
        "Class", "Alcohol", "Malic acid", "Ash", "Alcalinity of ash",
        "Magnesium", "Total phenols", "Flavanoids", "Nonflavanoid phenols",
        "Proanthocyanins", "Color intensity", "Hue",
        "OD280/OD315 of diluted wines", "Proline",
    ],
)
wine_data.head()


In [ ]:
wine_data["Class"].unique()


In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(wine_data.astype(float).corr(), square=True, annot=True,
            cmap=plt.cm.RdBu, linewidths=0.1)
plt.title("Pearson Correlation of Features")
plt.show()


In [ ]:
sns.pairplot(wine_data.sample(120, random_state=42), hue="Class")
plt.show()


## Prapemrosesan

Fitur distandarkan agar SVM dan KNN, yang bergantung pada jarak, bekerja seimbang.


In [ ]:
feature_columns = list(wine_data.columns)[1:]
target = wine_data["Class"]
features = preprocessing.StandardScaler().fit_transform(wine_data[feature_columns])

print("Mean:", np.round(features.mean(axis=0), 6)[:5])
print("Std:", np.round(features.std(axis=0), 6)[:5])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, random_state=1
)
print("Train set", X_train.shape, y_train.shape)
print("Test set", X_test.shape, y_test.shape)


## Membangun stacking classifier

Model dasar yang dipakai adalah SVM, KNN, dan pohon keputusan. Regresi logistik bertindak sebagai model meta.


In [ ]:
base_learners = [
    ("SVM", SVC(random_state=42)),
    ("KNN", KNeighborsClassifier()),
    ("dt", DecisionTreeClassifier(random_state=42)),
]

stacked_model = StackingClassifier(
    estimators=base_learners, final_estimator=LogisticRegression(max_iter=1000)
).fit(X_train, y_train)
stacked_model


In [ ]:
predictions = stacked_model.predict(X_test)
get_accuracy(X_train, X_test, y_train, y_test, stacked_model)


Stacking cenderung lebih mudah overfitting daripada model tunggal. Karena itu evaluasi harus memakai data uji yang tidak pernah dilihat model.


## Latihan: mengganti pohon keputusan dengan SVM kernel linear

Susunan model dasar diubah agar dua di antaranya memakai SVM dengan kernel berbeda.


In [ ]:
alternative_learners = [
    ("SVM_rbf", SVC(random_state=42)),
    ("SVM_linear", SVC(kernel="linear", random_state=42)),
    ("KNN", KNeighborsClassifier()),
]

alternative_model = StackingClassifier(
    estimators=alternative_learners, final_estimator=LogisticRegression(max_iter=1000)
).fit(X_train, y_train)
get_accuracy(X_train, X_test, y_train, y_test, alternative_model)


## Grid search pada stacking classifier

Hyperparameter model dasar diakses lewat awalan nama, misalnya `SVM__C` atau `knn__n_neighbors`.


In [ ]:
DRUG_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "drug200.csv"
)
drug_data = pd.read_csv(DRUG_URL, delimiter=",")

drug_features = drug_data[["Age", "Sex", "BP", "Cholesterol", "Na_to_K"]].copy()
drug_features["Sex"] = drug_features["Sex"].map({"F": 0, "M": 1})
drug_features["BP"] = drug_features["BP"].map({"LOW": 0, "NORMAL": 1, "HIGH": 2})
drug_features["Cholesterol"] = drug_features["Cholesterol"].map(
    {"NORMAL": 0, "HIGH": 1}
)
drug_features = preprocessing.StandardScaler().fit_transform(drug_features)

X_train, X_test, y_train, y_test = train_test_split(
    drug_features, drug_data["Drug"], test_size=0.3, random_state=4
)
print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)


In [ ]:
stacked_drug = StackingClassifier(
    estimators=[
        ("SVM", SVC(random_state=42)),
        ("knn", KNeighborsClassifier()),
        ("dt", DecisionTreeClassifier(random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=1000),
)

param_grid = {
    "dt__max_depth": [3, 5, 8],
    "SVM__C": [0.1, 1, 10],
    "SVM__kernel": ["linear", "rbf"],
    "knn__n_neighbors": [1, 4, 8],
}

search = GridSearchCV(
    estimator=stacked_drug, param_grid=param_grid, scoring="accuracy", cv=3, n_jobs=-1
).fit(X_train, y_train)
search.best_score_


In [ ]:
search.best_params_


In [ ]:
get_accuracy(X_train, X_test, y_train, y_test, search.best_estimator_)


## Ringkasan

Stacking memberi akurasi tinggi karena menggabungkan kekuatan beberapa model, tetapi biaya latihnya lebih besar dan risikonya overfitting lebih tinggi. Grid search membantu memilih kombinasi hyperparameter yang masuk akal.
